In [14]:
import numpy as np
import pandas as pd
import kagglehub
from pathlib import Path as pth


# Download latest version
path = kagglehub.dataset_download("simaanjali/diabetes-classification-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'diabetes-classification-dataset' dataset.
Path to dataset files: /kaggle/input/diabetes-classification-dataset


In [15]:
pa = pth("/kaggle/input/diabetes-classification-dataset")
for i in pth.iterdir(pa) :
    print(i)

pa = pth("/kaggle/input/diabetes-classification-dataset/Diabetes Classification.csv")
df = pd.read_csv(pa)
df = pd.DataFrame(df)
df.isna().sum()
df.drop('Unnamed: 0',axis=1, inplace=True)
df

/kaggle/input/diabetes-classification-dataset/Diabetes Classification.csv


,Age,Gender,BMI,Chol,TG,HDL,LDL,Cr,BUN,Diagnosis
0,50,F,24,4.20,0.90,2.40,1.40,46.0,4.70,0
1,26,M,23,3.70,1.40,1.10,2.10,62.0,4.50,0
2,33,M,21,4.90,1.00,0.80,2.00,46.0,7.10,0
3,45,F,21,2.90,1.00,1.00,1.50,24.0,2.30,0
4,50,F,24,3.60,1.30,0.90,2.10,50.0,2.00,0
...,...,...,...,...,...,...,...,...,...,...
5127,54,M,23,5.00,1.50,1.24,2.98,77.0,3.50,1
5128,50,F,22,4.37,2.09,1.37,2.29,47.3,4.40,1
5129,67,M,24,3.89,1.38,1.14,2.17,70.6,4.73,1
5130,60,F,29,5.91,1.29,1.73,2.85,50.2,7.33,1


In [ ]:
y = df['Diagnosis']
y =np.array(y)

x = df.iloc[:,:-1]
x["Gender"] = x["Gender"].map({"M":1,"F":0})
x = np.array(x)
x

array([[50.  ,  0.  , 24.  , ...,  1.4 , 46.  ,  4.7 ],
       [26.  ,  1.  , 23.  , ...,  2.1 , 62.  ,  4.5 ],
       [33.  ,  1.  , 21.  , ...,  2.  , 46.  ,  7.1 ],
       ...,
       [67.  ,  1.  , 24.  , ...,  2.17, 70.6 ,  4.73],
       [60.  ,  0.  , 29.  , ...,  2.85, 50.2 ,  7.33],
       [37.  ,  1.  , 34.  , ...,  2.87, 75.5 ,  4.61]])

In [ ]:
def tree(x,y) :

    best_gain = -1
    best_col = None
    best_cut = None

    counts = np.bincount(y, minlength=2)
    p0 = counts[0] / len(y)
    p1 = counts[1] / len(y)

    total_gini = 1-(p0**2 + p1**2)

    if best_col is None:

        return np.argmax(np.bincount(y))

    for o in range(x.shape[1]) :

        sorted_col = np.sort(x[:, o])

        averages = (sorted_col[1:] + sorted_col[:-1]) / 2

        for i in np.unique(averages) :

            y_right = y[x[:,o] > i]
            y_left = y[x[:,o] <= i]

            if len(y_right) > 0 and len(y_left) > 0:

                p_right_1 = np.sum(y_right==1)/len(y_right)
                p_right_0 = np.sum(y_right==0)/len(y_right)

                p_left_1 = np.sum(y_left==1)/len(y_left)
                p_left_0 = np.sum(y_left==0)/len(y_left)

                gini_right = 1-(p_right_0**2 + p_right_1**2)
                gini_left = 1-(p_left_0**2 + p_left_1**2)

                w_r = len(y_right) / len(y)
                w_l = len(y_left) / len(y)

                gain = total_gini - (w_r * gini_right + w_l * gini_left)

                if gain > best_gain :
                    best_gain = gain
                    best_col = o
                    best_cut = i

    x_left = x[x[:, best_col] <= best_cut]
    x_right = x[x[:, best_col] > best_cut]

    left_y = y[x[:, best_col] <= best_cut]
    right_y = y[x[:, best_col] > best_cut]

    return {
        'col': best_col,
        'cut': best_cut,
        'left': tree(x_left, left_y),
        'right': tree(x_right, right_y)
}

In [ ]:
my_model = tree(x,y)
def predict(my_model, patient):
    if not isinstance(my_model, dict):
        return my_model
    col = my_model['col']
    cut = my_model['cut']

    if patient[col] <= cut:
        return predict(my_model['left'], patient)
    else:
        return predict(my_model['right'], patient)

In [25]:
test = x[:100,:]*200
test = test*500
test_predictions = [predict(my_model, patient) for patient in test]

print(test_predictions)

[np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)